# Module 7: Final Demo App — Redis Caching for AI

## What You're Building

This is the capstone of the course. We build a **Gradio web app** that brings together everything you've learned:

| Tab | Feature Demonstrated |
|-----|---------------------|
| **Exact Cache** | Ask the same question twice — see cache hit vs miss side by side |
| **Semantic Cache** | Similar questions share cached answers |
| **Agent Memory** | A chatbot that remembers your conversation across messages |
| **RAG Demo** | Ask questions about TechCorp — answered from documents |
| **Cache Dashboard** | Live view of what's stored in Redis, stats, and cleanup |

---

Run **all cells in order**, then the final cell launches the app in your browser.

In [49]:
import redis
import openai
import hashlib
import json
import time
import os
import numpy as np
import gradio as gr
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# ── Connections ──────────────────────────────────────
r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    decode_responses=True
)

openai_client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
groq_client   = Groq(api_key=os.getenv('GROQ_API_KEY'))

r.ping()
print("All systems connected — Redis, OpenAI, Groq")

All systems connected — Redis, OpenAI, Groq


In [50]:
# ── Shared helpers ───────────────────────────────────

def make_exact_key(prompt: str) -> str:
    return f"app:exact:{hashlib.md5(prompt.strip().lower().encode()).hexdigest()}"

def get_embedding(text: str) -> list:
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=text.strip()
    )
    return response.data[0].embedding

def cosine_similarity(a: list, b: list) -> float:
    a_arr, b_arr = np.array(a), np.array(b)
    return float(np.dot(a_arr, b_arr) / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr)))

COST_INPUT  = 0.00000015   # gpt-4o-mini input cost per token
COST_OUTPUT = 0.00000060   # gpt-4o-mini output cost per token

print("Helpers defined!")

Helpers defined!


In [51]:
# ── TAB 1: Exact Match Cache ─────────────────────────

def exact_cache_ask(prompt: str):
    """Ask with exact match caching. Returns (response_text, metadata_text)."""
    if not prompt.strip():
        return "", "Please enter a question."

    start = time.time()
    key = make_exact_key(prompt)
    cached = r.get(key)

    if cached:
        data = json.loads(cached)
        elapsed_ms = (time.time() - start) * 1000
        meta = (
            f"Source     : CACHE HIT\n"
            f"Time       : {elapsed_ms:.1f} ms\n"
            f"Cost       : $0.000000 (FREE!)\n"
            f"Redis key  : {key}\n"
            f"TTL        : {r.ttl(key)} seconds remaining"
        )
        return data["response"], meta

    # Cache miss
    api_response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )
    elapsed_ms = (time.time() - start) * 1000
    response_text = api_response.choices[0].message.content
    in_tok  = api_response.usage.prompt_tokens
    out_tok = api_response.usage.completion_tokens
    cost = (in_tok * COST_INPUT) + (out_tok * COST_OUTPUT)

    # Store with 1-hour TTL
    r.set(key, json.dumps({"response": response_text, "prompt": prompt}), ex=3600)

    meta = (
        f"Source     : API CALL (now cached!)\n"
        f"Time       : {elapsed_ms:.0f} ms\n"
        f"Cost       : ${cost:.6f}\n"
        f"Tokens     : {in_tok} in / {out_tok} out\n"
        f"Redis key  : {key}"
    )
    return response_text, meta

print("Tab 1 functions ready")

Tab 1 functions ready


In [52]:
# ── TAB 2: Semantic Cache ────────────────────────────

SEM_PREFIX   = "app:sem:"
SEM_INDEX    = "app:sem:index"
SEM_THRESHOLD = 0.70

def semantic_cache_ask(prompt: str, threshold: float = SEM_THRESHOLD):
    """Ask with semantic caching."""
    if not prompt.strip():
        return "", "Please enter a question."

    start = time.time()

    # Get embedding for the query
    query_emb = get_embedding(prompt)

    # Search semantic cache
    entry_ids = r.smembers(SEM_INDEX)
    best_score, best_entry = 0.0, None
    for eid in entry_ids:
        data = r.hgetall(f"{SEM_PREFIX}{eid}")
        if not data:
            continue
        score = cosine_similarity(query_emb, json.loads(data["embedding"]))
        if score > best_score:
            best_score, best_entry = score, data

    if best_score >= threshold and best_entry:
        elapsed_ms = (time.time() - start) * 1000
        meta = (
            f"Source          : SEMANTIC CACHE HIT\n"
            f"Similarity      : {best_score:.4f} (threshold: {threshold})\n"
            f"Matched question: {best_entry['prompt']}\n"
            f"Time            : {elapsed_ms:.0f} ms\n"
            f"Cost            : $0.000000 (FREE!)"
        )
        return best_entry["response"], meta

    # Miss — call API
    api_response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )
    elapsed_ms = (time.time() - start) * 1000
    response_text = api_response.choices[0].message.content
    in_tok  = api_response.usage.prompt_tokens
    out_tok = api_response.usage.completion_tokens
    cost = (in_tok * COST_INPUT) + (out_tok * COST_OUTPUT)

    # Store in semantic cache
    eid = hashlib.md5(prompt.encode()).hexdigest()
    r.hset(f"{SEM_PREFIX}{eid}", mapping={
        "prompt": prompt,
        "response": response_text,
        "embedding": json.dumps(query_emb)
    })
    r.sadd(SEM_INDEX, eid)

    meta = (
        f"Source     : API CALL (stored in semantic cache)\n"
        f"Best match : {best_score:.4f} (below threshold {threshold})\n"
        f"Time       : {elapsed_ms:.0f} ms\n"
        f"Cost       : ${cost:.6f}\n"
        f"Tokens     : {in_tok} in / {out_tok} out"
    )
    return response_text, meta

print("Tab 2 functions ready")

Tab 2 functions ready


In [53]:
# ── TAB 3: Agent Memory Chat ─────────────────────────

APP_SESSION_ID = "app:demo:session"

def memory_chat(user_message: str, history: list):
    """Gradio chatbot function — maintains history in Redis."""
    if not user_message.strip():
        return history, ""

    key = f"conversation:{APP_SESSION_ID}"

    # Add user message to Redis
    r.rpush(key, json.dumps({"role": "user", "content": user_message, "ts": time.time()}))
    r.ltrim(key, -20, -1)  # Keep last 20 messages
    r.expire(key, 7200)

    # Build message list from Redis history
    raw = r.lrange(key, 0, -1)
    messages = [
        {"role": "system", "content": "You are a helpful and concise AI assistant. Remember everything the user tells you in this conversation."}
    ]
    messages.extend([
        {"role": m["role"], "content": m["content"]}
        for m in [json.loads(x) for x in raw]
    ])

    # Call Groq
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        max_tokens=400
    )
    reply = response.choices[0].message.content

    # Store assistant reply in Redis
    r.rpush(key, json.dumps({"role": "assistant", "content": reply, "ts": time.time()}))
    r.ltrim(key, -20, -1)
    r.expire(key, 7200)

    # Update Gradio chat history display
    history.append({"role": "user", "content": user_message})
    history.append({"role": "assistant", "content": reply})
    return history, ""

def clear_memory_chat():
    r.delete(f"conversation:{APP_SESSION_ID}")
    return [], "Memory cleared! Start a new conversation."

def get_memory_status():
    key = f"conversation:{APP_SESSION_ID}"
    count = r.llen(key)
    ttl = r.ttl(key)
    return f"Messages in Redis: {count}  |  Session expires in: {ttl}s  |  Key: {key}"

print("Tab 3 functions ready")

Tab 3 functions ready


In [54]:
# ── TAB 4: RAG System ────────────────────────────────

RAG_DOCS = [
    "TechCorp was founded in 2018 by Sarah Chen and Marcus Torres in San Francisco.",
    "DataMind offers three plans: Starter ($29/month, 5 users), Professional ($99/month, 50 users), and Enterprise (custom).",
    "Starter and Professional plans have a 14-day money-back guarantee. Enterprise contracts are non-refundable after 30 days.",
    "TechCorp's customer support is 24/7 for Enterprise, and 9am-6pm PST Monday-Friday for other plans.",
    "DataMind integrates with Salesforce, HubSpot, Slack, Google Analytics, and over 50 other tools.",
    "TechCorp raised $45 million in Series B funding in March 2023, led by Sequoia Capital.",
    "Data is stored in AWS us-east-1 and eu-west-1 regions. Enterprise can use on-premise deployment.",
    "All data is encrypted at rest with AES-256 and in transit with TLS 1.3.",
    "TechCorp is SOC 2 Type II certified and GDPR compliant as of January 2024.",
    "The Professional plan includes API access, advanced analytics, and priority support.",
]

# Pre-compute and cache document embeddings
print("Pre-computing RAG document embeddings...")
rag_doc_embeddings = {}
for doc in RAG_DOCS:
    emb_key = f"app:embedding:{hashlib.sha256(doc.encode()).hexdigest()[:16]}"
    cached = r.get(emb_key)
    if cached:
        rag_doc_embeddings[doc] = json.loads(cached)
    else:
        emb = get_embedding(doc)
        r.set(emb_key, json.dumps(emb))  # Permanent — embeddings don't change
        rag_doc_embeddings[doc] = emb
print(f"Done! {len(RAG_DOCS)} documents indexed.")


def rag_answer(question: str):
    if not question.strip():
        return "", "Please enter a question."

    # Check answer cache
    answer_key = f"app:rag:answer:{hashlib.md5(question.strip().lower().encode()).hexdigest()}"
    cached = r.get(answer_key)
    if cached:
        data = json.loads(cached)
        meta = (
            f"Source   : ANSWER CACHE HIT\n"
            f"Time     : < 1 ms\n"
            f"Cost     : $0.000000 (FREE!)\n"
            f"TTL      : {r.ttl(answer_key)} seconds"
        )
        return data["answer"], meta

    start = time.time()

    # Retrieve relevant documents
    q_emb = get_embedding(question)
    scored = [
        (cosine_similarity(q_emb, emb), doc)
        for doc, emb in rag_doc_embeddings.items()
    ]
    scored.sort(reverse=True)
    top_docs = [doc for _, doc in scored[:3]]

    context = "\n".join([f"- {d}" for d in top_docs])
    prompt = f"""You are TechCorp's support assistant. Answer using ONLY the provided information. If you can't answer from it, say so.

Information:
{context}

Question: {question}
Answer:"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )
    answer = response.choices[0].message.content
    elapsed_ms = (time.time() - start) * 1000

    # Cache the answer for 1 hour
    r.set(answer_key, json.dumps({"question": question, "answer": answer}), ex=3600)

    meta = (
        f"Source        : COMPUTED + CACHED\n"
        f"Time          : {elapsed_ms:.0f} ms\n"
        f"Top sources   :\n" +
        "\n".join([f"  - {d[:70]}..." for d in top_docs])
    )
    return answer, meta

print("Tab 4 functions ready")

Pre-computing RAG document embeddings...
Done! 10 documents indexed.
Tab 4 functions ready


In [55]:
# ── TAB 5: Dashboard ─────────────────────────────────

def get_dashboard():
    """Get a snapshot of everything currently in Redis."""
    exact_keys   = r.keys("app:exact:*")
    sem_keys     = r.keys("app:sem:*")
    emb_keys     = r.keys("app:embedding:*")
    rag_keys     = r.keys("app:rag:*")
    conv_keys    = r.keys("conversation:*")
    rate_keys    = r.keys("rate:*")

    info = r.info()
    total = r.dbsize()

    lines = [
        "=" * 50,
        "REDIS CACHE DASHBOARD",
        "=" * 50,
        f"Redis version      : {info['redis_version']}",
        f"Memory used        : {info['used_memory_human']}",
        f"Total keys in DB   : {total}",
        "",
        "App cache breakdown:",
        f"  Exact match cache     : {len(exact_keys)} entries",
        f"  Semantic cache        : {len([k for k in sem_keys if 'index' not in k])} entries",
        f"  Embedding cache       : {len(emb_keys)} entries",
        f"  RAG answer cache      : {len(rag_keys)} entries",
        f"  Conversation sessions : {len(conv_keys)} sessions",
        f"  Rate limiter keys     : {len(rate_keys)} entries",
    ]

    if exact_keys:
        lines.append("")
        lines.append("Exact cache entries:")
        for key in exact_keys[:5]:
            data = r.get(key)
            if data:
                prompt = json.loads(data).get("prompt", "?")[:60]
                ttl = r.ttl(key)
                lines.append(f"  [{ttl:>5}s] {prompt}")

    if conv_keys:
        lines.append("")
        lines.append("Active conversations:")
        for key in conv_keys:
            count = r.llen(key)
            ttl = r.ttl(key)
            lines.append(f"  {key}  ({count} msgs, expires in {ttl}s)")

    return "\n".join(lines)


def clear_all_caches():
    patterns = ["app:exact:*", "app:sem:*", "app:rag:*"]
    total_deleted = 0
    for pattern in patterns:
        keys = r.keys(pattern)
        if keys:
            r.delete(*keys)
            total_deleted += len(keys)
    return f"Cleared {total_deleted} cache entries (kept embeddings and conversations)"


print("Tab 5 functions ready")

Tab 5 functions ready


In [56]:
# ── Build the Gradio App ─────────────────────────────

with gr.Blocks(
    title="Redis AI Caching — Course Demo",
) as demo:

    gr.Markdown("""
    # Redis Caching for AI & LLMs — Interactive Demo
    Everything from this course, running live. Try each tab!
    """)

    # ── Tab 1: Exact Match Cache ──────────────────────
    with gr.Tab("Exact Cache"):
        gr.Markdown("""
        ## Exact Match Caching (Module 2)
        Ask any question. Then ask the **exact same question again** — watch the source change to CACHE HIT
        and the time drop from seconds to under 1 millisecond.

        **Try it:** Ask "What is a vector database?" twice.
        """)
        with gr.Row():
            exact_input = gr.Textbox(
                label="Your question",
                placeholder="e.g. What is a vector database?",
                lines=2
            )
        exact_btn = gr.Button("Ask", variant="primary")
        with gr.Row():
            exact_response = gr.Textbox(label="Response", lines=8)
            exact_meta     = gr.Textbox(label="Cache Info", lines=8)
        exact_btn.click(
            fn=exact_cache_ask,
            inputs=[exact_input],
            outputs=[exact_response, exact_meta]
        )

    # ── Tab 2: Semantic Cache ─────────────────────────
    with gr.Tab("Semantic Cache"):
        gr.Markdown("""
        ## Semantic Caching (Module 3)
        Ask a question. Then ask a **similar but differently worded** question.
        If the similarity is above the threshold, you'll get a SEMANTIC CACHE HIT.

        **Try it:** First ask "What is machine learning?", then ask "Can you explain ML to me?"
        """)
        with gr.Row():
            sem_input     = gr.Textbox(label="Your question", placeholder="e.g. What is machine learning?", lines=2)
            sem_threshold = gr.Slider(
                minimum=0.7, maximum=0.99, value=0.70, step=0.01,
                label="Similarity threshold (lower = more cache hits but less accurate)"
            )
        sem_btn = gr.Button("Ask", variant="primary")
        with gr.Row():
            sem_response = gr.Textbox(label="Response", lines=8)
            sem_meta     = gr.Textbox(label="Cache Info", lines=8)
        sem_btn.click(
            fn=semantic_cache_ask,
            inputs=[sem_input, sem_threshold],
            outputs=[sem_response, sem_meta]
        )

    # ── Tab 3: Agent Memory ───────────────────────────
    with gr.Tab("Agent Memory"):
        gr.Markdown("""
        ## AI Agent Conversation Memory (Module 4)
        Chat with an AI that **remembers** your entire conversation.
        The conversation is stored in Redis — not in the browser or any in-memory variable.

        **Try it:** Tell the bot your name and what you do. Then ask "What is my name?"
        """)
        chatbot = gr.Chatbot(label="Chat (powered by Groq + Redis Memory)", height=400)
        with gr.Row():
            chat_input = gr.Textbox(label="Your message", placeholder="Say something...", lines=1, scale=4)
            chat_send  = gr.Button("Send", variant="primary", scale=1)
        mem_status = gr.Textbox(label="Redis Memory Status", lines=1, interactive=False)
        clear_btn  = gr.Button("Clear Memory & Start Over", variant="secondary")

        chat_send.click(
            fn=memory_chat,
            inputs=[chat_input, chatbot],
            outputs=[chatbot, chat_input]
        ).then(
            fn=get_memory_status,
            outputs=[mem_status]
        )
        chat_input.submit(
            fn=memory_chat,
            inputs=[chat_input, chatbot],
            outputs=[chatbot, chat_input]
        ).then(
            fn=get_memory_status,
            outputs=[mem_status]
        )
        clear_btn.click(
            fn=clear_memory_chat,
            outputs=[chatbot, mem_status]
        )

    # ── Tab 4: RAG ────────────────────────────────────
    with gr.Tab("RAG Demo"):
        gr.Markdown("""
        ## RAG with Cached Embeddings (Module 6)
        Ask questions about **TechCorp** — a fictional company.
        The answers are grounded in the knowledge base. Repeated questions are served from the answer cache.

        **Try asking:**
        - "What is the refund policy?"
        - "How many users does the Professional plan support?"
        - "Is TechCorp GDPR compliant?"
        - "What is the refund policy?" (again — should be CACHE HIT!)
        """)
        rag_input = gr.Textbox(
            label="Question about TechCorp",
            placeholder="e.g. What is the refund policy?",
            lines=2
        )
        rag_btn = gr.Button("Ask TechCorp Support", variant="primary")
        with gr.Row():
            rag_response = gr.Textbox(label="Answer", lines=8)
            rag_meta     = gr.Textbox(label="Cache & Source Info", lines=8)
        rag_btn.click(
            fn=rag_answer,
            inputs=[rag_input],
            outputs=[rag_response, rag_meta]
        )

    # ── Tab 5: Dashboard ──────────────────────────────
    with gr.Tab("Cache Dashboard"):
        gr.Markdown("""
        ## Live Redis Dashboard (Module 1)
        See everything currently stored in Redis — cache entries, conversations, and server stats.
        Refresh anytime to see updates.
        """)
        dashboard_output = gr.Textbox(label="Redis Contents", lines=25, interactive=False)
        with gr.Row():
            refresh_btn    = gr.Button("Refresh Dashboard", variant="primary")
            clear_cache_btn = gr.Button("Clear App Caches (keep embeddings)", variant="secondary")
        clear_result = gr.Textbox(label="Clear result", lines=1, interactive=False)

        refresh_btn.click(fn=get_dashboard, outputs=[dashboard_output])
        clear_cache_btn.click(fn=clear_all_caches, outputs=[clear_result]).then(
            fn=get_dashboard, outputs=[dashboard_output]
        )

        # Auto-load dashboard on tab open
        demo.load(fn=get_dashboard, outputs=[dashboard_output])

print("Gradio app built!")
print("Run the next cell to launch it.")

Gradio app built!
Run the next cell to launch it.


In [57]:
# Launch the app!
# Opens at http://127.0.0.1:7860 in your browser automatically
demo.launch(
    server_name="127.0.0.1",
    server_port=7863,
    share=False,      # Set share=True to get a public URL for 72 hours
    inbrowser=True,   # Automatically open browser
    theme=gr.themes.Soft(primary_hue="blue")
)

OSError: Cannot find empty port in range: 7863-7863. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

## What to Try in the App

### Tab 1 — Exact Cache:
1. Ask: **"What is gradient descent?"**
2. Look at the cache info — it says API CALL
3. Ask the **exact same question** again
4. Observe: CACHE HIT, sub-millisecond response, $0 cost

### Tab 2 — Semantic Cache:
1. Ask: **"What is machine learning?"**
2. Ask: **"Can you explain machine learning to me?"**
3. Ask: **"Explain ML"**
4. Watch all three resolve to the same cached answer via semantic similarity

### Tab 3 — Agent Memory:
1. Say: **"Hi! My name is [your name] and I'm a data scientist"**
2. Ask a few unrelated questions
3. Then ask: **"What is my name?"** and **"What do I do?"**
4. The agent remembers — because Redis stored the whole conversation!

### Tab 4 — RAG:
1. Ask: **"What is the refund policy?"**
2. Ask: **"What integrations does DataMind support?"**
3. Ask the **refund policy question again** — it's now an instant CACHE HIT

### Tab 5 — Dashboard:
1. Hit Refresh to see all your cached entries
2. After exploring the other tabs, you'll see the cache fill up
3. Use "Clear App Caches" to reset (keeps embeddings — they're permanent)

---

## Course Complete!

Congratulations on finishing the **Redis Caching for AI & LLMs** course!

### What You've Learned:

| Module | Skill |
|--------|-------|
| 1 | Redis fundamentals: strings, hashes, lists, TTL |
| 2 | Exact match LLM caching — hash prompts, store responses |
| 3 | Semantic caching — embeddings + cosine similarity |
| 4 | Agent memory — conversation history, sliding window, user profiles |
| 5 | Rate limiting — fixed window, sliding window, token budgets |
| 6 | RAG caching — embedding, retrieval, and answer cache layers |
| 7 | Production-style Gradio app tying it all together |

### What's Next?
- **Redis Vector Search** — Use built-in HNSW index instead of brute-force cosine similarity (scales to millions of documents)
- **Redis Pub/Sub** — Stream LLM tokens in real-time to multiple clients
- **LangChain Redis integration** — `RedisCache` and `RedisChatMessageHistory` classes
- **Production deployment** — Redis Sentinel (HA) or Redis Cluster for scale